# 2. BC카드 매출 효과 분석 (주력)

**데이터**: BC카드 주간/월간 매출 (2025.01~2026.04)
**방법**: CausalImpact + DID 교차검증
**핵심**: 방송 촬영 읍면동 vs 비촬영 읍면동 매출 변화

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from config import CARD_DIR, BROADCAST_DONG_MAP, ALL_TREATED_DONGS, read_csv_auto, load_monthly_data

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 2.1 데이터 로드 & 읍면동 목록 파악

**외지인 기준 소비**를 핵심 지표로 사용:
- `AS_WEEK_CCND_EXCL_RSDT_CRTR_CCND_CSPT_DONG`: 주간, 비거주자(외지인) 기준, 동 단위 소비
- 외지인 소비 = 관광객 소비 proxy

In [ ]:
# 외지인 기준 주간 동 단위 매출 로드
df_week = load_monthly_data(CARD_DIR, 'AS_WEEK_CCND_EXCL_RSDT_CRTR_CCND_CSPT_DONG')
print(f"로드 완료: {df_week.shape}")
print(f"\n컬럼: {list(df_week.columns)}")
df_week.head()

In [ ]:
# 읍면동 목록 확인
dong_col = [c for c in df_week.columns if 'DONG_NM' in c and 'FRCS' in c]
if dong_col:
    dong_col = dong_col[0]
else:
    dong_col = [c for c in df_week.columns if 'DONG' in c.upper()][0]

all_dongs = sorted(df_week[dong_col].dropna().unique())
print(f"전체 읍면동 ({len(all_dongs)}개):")
for d in all_dongs:
    marker = " ★ 촬영지" if any(t in str(d) for t in ALL_TREATED_DONGS) else ""
    print(f"  {d}{marker}")

# 통제군 읍면동 식별
control_dongs = [d for d in all_dongs if not any(t in str(d) for t in ALL_TREATED_DONGS)]
treated_dongs = [d for d in all_dongs if any(t in str(d) for t in ALL_TREATED_DONGS)]
print(f"\n처치군: {treated_dongs}")
print(f"통제군: {control_dongs}")

## 2.2 주간 매출 시계열 구축

In [ ]:
# 시간 컬럼 파악 (CRTR_YM + 주차)
time_cols = [c for c in df_week.columns if 'YM' in c or 'WEEK' in c.upper() or 'WK' in c.upper() or 'CRTR' in c]
print(f"시간 관련 컬럼: {time_cols}")

# 매출액 컬럼 파악
amt_cols = [c for c in df_week.columns if 'AMT' in c.upper()]
print(f"매출액 컬럼: {amt_cols}")

# 건수 컬럼
cnt_cols = [c for c in df_week.columns if 'NOCS' in c.upper() or 'CNT' in c.upper()]
print(f"건수 컬럼: {cnt_cols}")

In [ ]:
# 주간 시계열 생성: 읍면동별 총 매출액 집계
# 주요 매출 컬럼 (ALL_USE_AMT 또는 전체 합계)
amt_col = 'ALL_USE_AMT' if 'ALL_USE_AMT' in df_week.columns else amt_cols[0]

# CRTR_YM이 정수형이면 문자열 변환
df_week['YM'] = df_week['CRTR_YM'].astype(str)

# 주차 컬럼이 있으면 사용, 없으면 CRTR_YM만 사용
week_col = [c for c in df_week.columns if 'WEEK' in c.upper() or 'WK' in c.upper()]
if week_col:
    df_week['period'] = df_week['YM'] + '_W' + df_week[week_col[0]].astype(str)
else:
    # 주간 데이터지만 주차 컬럼이 별도 없으면 시작일 기준
    date_col = [c for c in df_week.columns if 'ST_DT' in c.upper() or 'STRT' in c.upper() or 'BGN' in c.upper()]
    if date_col:
        df_week['period'] = df_week[date_col[0]].astype(str)
    else:
        df_week['period'] = df_week['YM']

print(f"기간 유니크: {sorted(df_week['period'].unique())[:10]} ...")
print(f"총 기간 수: {df_week['period'].nunique()}")

In [ ]:
# 읍면동 그룹별 주간 매출 합계
df_week['group'] = df_week[dong_col].apply(
    lambda x: 'treated' if any(t in str(x) for t in ALL_TREATED_DONGS) else 'control'
)

weekly_ts = df_week.groupby(['period', 'group'])[amt_col].sum().unstack('group').fillna(0)
weekly_ts = weekly_ts.sort_index()

print(f"시계열 shape: {weekly_ts.shape}")
weekly_ts.head(10)

## 2.3 시계열 시각화: 처치군 vs 통제군

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

# 절대값
ax1 = axes[0]
weekly_ts.plot(ax=ax1)
ax1.set_title('주간 외지인 매출: 촬영 읍면동 vs 비촬영 읍면동')
ax1.set_ylabel('매출액')
ax1.legend(['통제군', '처치군'])

# 방송 시점 표시
broadcast_dates = {
    '전국노래자랑': '2025-06',
    '11월캠페인': '2025-11',
    '뛰어야산다2': '2026-01',
}
for name, ym in broadcast_dates.items():
    matches = [p for p in weekly_ts.index if ym in str(p)]
    if matches:
        idx = weekly_ts.index.get_loc(matches[0])
        for ax in axes:
            ax.axvline(idx, color='red', linestyle='--', alpha=0.7)
            ax.text(idx, ax.get_ylim()[1]*0.95, name, fontsize=8, rotation=90, va='top', color='red')

# 비율 (처치군/통제군)
ax2 = axes[1]
ratio = weekly_ts['treated'] / weekly_ts['control'].replace(0, np.nan)
ratio.plot(ax=ax2, color='purple')
ax2.axhline(ratio.mean(), color='gray', linestyle=':', alpha=0.5)
ax2.set_title('처치군/통제군 매출 비율 (방송 효과 시 상승 기대)')
ax2.set_ylabel('비율')

plt.tight_layout()
plt.show()

## 2.4 업종별 분해

In [ ]:
# 업종 컬럼 확인
biz_col = [c for c in df_week.columns if 'TOBIZ' in c and 'NM' in c]
if biz_col:
    biz_col = biz_col[0]
    print(f"업종 컬럼: {biz_col}")
    print(f"\n상위 20개 업종:")
    top_biz = df_week.groupby(biz_col)[amt_col].sum().sort_values(ascending=False).head(20)
    display(top_biz)
else:
    print("업종 컬럼 없음 - 다른 파일 사용 필요")

In [ ]:
# 관광 관련 업종 필터링
tourism_keywords = ['숙박', '호텔', '모텔', '펜션', '한식', '일식', '중식', '양식', '카페',
                     '관광', '온천', '여행', '레저', '주유소', '편의점', '분식', '치킨',
                     '제과', '패스트', '주차']

if biz_col:
    df_week['is_tourism'] = df_week[biz_col].apply(
        lambda x: any(k in str(x) for k in tourism_keywords)
    )
    
    # 관광업종만 처치/통제 비교
    tourism_ts = df_week[df_week['is_tourism']].groupby(['period', 'group'])[amt_col].sum().unstack('group').fillna(0)
    general_ts = df_week[~df_week['is_tourism']].groupby(['period', 'group'])[amt_col].sum().unstack('group').fillna(0)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    if 'treated' in tourism_ts.columns and 'control' in tourism_ts.columns:
        (tourism_ts['treated'] / tourism_ts['control'].replace(0, np.nan)).plot(ax=axes[0], color='#e74c3c')
        axes[0].set_title('관광 업종 (처치/통제 비율)')
        axes[0].axhline((tourism_ts['treated'] / tourism_ts['control'].replace(0, np.nan)).mean(),
                        color='gray', linestyle=':', alpha=0.5)
    
    if 'treated' in general_ts.columns and 'control' in general_ts.columns:
        (general_ts['treated'] / general_ts['control'].replace(0, np.nan)).plot(ax=axes[1], color='#3498db')
        axes[1].set_title('일반 업종 (처치/통제 비율) - 플라시보')
        axes[1].axhline((general_ts['treated'] / general_ts['control'].replace(0, np.nan)).mean(),
                        color='gray', linestyle=':', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    print("일반 업종은 방송 효과가 없어야 정상 (플라시보 테스트)")

## 2.5 CausalImpact 분석

```
pip install pycausalimpact
```

In [ ]:
try:
    from causalimpact import CausalImpact
    HAS_CI = True
except ImportError:
    print("pycausalimpact 미설치. pip install pycausalimpact 실행 후 재시도")
    HAS_CI = False

In [ ]:
def run_causal_impact(treated_series, control_series, pre_period, post_period, title=""):
    """
    CausalImpact 실행 및 결과 시각화
    
    Args:
        treated_series: 처치군 시계열 (pandas Series, DatetimeIndex)
        control_series: 통제군 시계열 (pandas Series, DatetimeIndex)
        pre_period: [시작, 끝] - 방송 전 기간
        post_period: [시작, 끝] - 방송 후 기간
        title: 차트 제목
    """
    if not HAS_CI:
        print("CausalImpact 미설치")
        return None
    
    data = pd.DataFrame({
        'y': treated_series,
        'x1': control_series,
    })
    data = data.dropna()
    
    ci = CausalImpact(data, pre_period, post_period)
    
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(ci.summary())
    print(ci.summary(output='report'))
    
    ci.plot()
    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()
    
    return ci

In [ ]:
# 분석 그룹 A: 전국노래자랑 (2025-06-08)
# pre: 2025-01 ~ 2025-05, post: 2025-06 ~ 2025-07
if HAS_CI and len(weekly_ts) > 0:
    # 인덱스를 날짜로 변환 (period → date 변환 필요)
    # 실제 실행 시 period 형식에 맞게 조정
    print("전국노래자랑 분석 준비...")
    print(f"period 샘플: {weekly_ts.index[:5].tolist()}")
    print("\n→ period를 날짜로 변환한 후 pre_period/post_period 설정 필요")
    print("→ 아래 셀에서 실제 period 형식 확인 후 수동 지정")

In [ ]:
# ★ 실행 시 period 형식 확인 후 아래 값을 수정할 것

# 예시: period가 '202501_W1' 형태인 경우
# pre_indices_A = [p for p in weekly_ts.index if p < '202506']
# post_indices_A = [p for p in weekly_ts.index if '202506' <= p <= '202507']

# 예시: period가 '20250106' (날짜) 형태인 경우  
# pre_indices_A = [p for p in weekly_ts.index if int(str(p)[:6]) < 202506]
# post_indices_A = [p for p in weekly_ts.index if 202506 <= int(str(p)[:6]) <= 202507]

# CausalImpact는 DatetimeIndex가 필요하므로 변환
# weekly_ts.index = pd.to_datetime(weekly_ts.index, format='...')

print("period 형식을 확인하고 위 주석을 해제하여 실행")

### 분석 그룹 B: 11월 방송 캠페인

In [ ]:
# 분석 그룹 B: 11월 캠페인 (2025-11 ~ 2025-12)
# pre: 2025-01 ~ 2025-10, post: 2025-11 ~ 2025-12
# 단, 6월 전국노래자랑 효과 잔류 가능 → pre를 2025-07부터로 조정 가능

if HAS_CI and len(weekly_ts) > 0:
    print("11월 캠페인 분석 준비...")
    print("pre: 2025-07 ~ 2025-10 (전국노래자랑 효과 소멸 후)")
    print("post: 2025-11 ~ 2025-12")

### 분석 그룹 C: 뛰어야산다2

In [ ]:
# 분석 그룹 C: 뛰어야산다2 (2026-01-12)
# pre: 2025-07 ~ 2025-10 (11월 캠페인 전)
# post: 2026-01 ~ 2026-02
# 주의: 11-12월 캠페인 효과 잔류 가능 → DID로 교차검증 필수

if HAS_CI and len(weekly_ts) > 0:
    print("뛰어야산다2 분석 준비...")
    print("주의: 같이삽시다(12/15 종영) → 뛰어야산다2(1/12) 간격 28일")
    print("→ 짧은 간격이므로 이전 효과 잔류 가능. DID 교차검증 권장")

## 2.6 DID (이중차분) 교차검증

In [ ]:
import statsmodels.formula.api as smf

def run_did(df, dong_col, amt_col, treatment_dongs, pre_periods, post_periods, title=""):
    """
    DID(이중차분) 회귀분석
    
    Y = b0 + b1*Treated + b2*Post + b3*(Treated*Post) + e
    b3 = 방송 효과 (DID 추정치)
    """
    df_did = df.copy()
    df_did['treated'] = df_did[dong_col].apply(
        lambda x: 1 if any(t in str(x) for t in treatment_dongs) else 0
    )
    df_did['post'] = df_did['period'].apply(
        lambda x: 1 if x in post_periods else (0 if x in pre_periods else np.nan)
    )
    df_did = df_did.dropna(subset=['post'])
    df_did['treated_post'] = df_did['treated'] * df_did['post']
    
    # 읍면동별 기간별 매출 합계
    agg = df_did.groupby([dong_col, 'period', 'treated', 'post', 'treated_post'])[amt_col].sum().reset_index()
    
    model = smf.ols(f'{amt_col} ~ treated + post + treated_post', data=agg).fit()
    
    print(f"\n{'='*60}")
    print(f"  DID: {title}")
    print(f"{'='*60}")
    print(model.summary().tables[1])
    print(f"\nDID 추정치 (treated_post): {model.params['treated_post']:,.0f}")
    print(f"p-value: {model.pvalues['treated_post']:.4f}")
    print(f"{'유의함 (p < 0.05)' if model.pvalues['treated_post'] < 0.05 else '유의하지 않음'}")
    
    return model

In [ ]:
# DID 실행 예시 (period 형식 확인 후 수정)
# pre_periods = [p for p in df_week['period'].unique() if ...]
# post_periods = [p for p in df_week['period'].unique() if ...]
# run_did(df_week, dong_col, amt_col, ALL_TREATED_DONGS, pre_periods, post_periods, "11월 캠페인")

print("period 형식 확인 후 위 코드 수정하여 실행")

## 2.7 플라시보 테스트

방송이 없었던 시점에 가짜 개입을 설정하여 효과가 나오지 않는지 확인

In [ ]:
# 플라시보: 2025-03을 가짜 개입 시점으로 설정
# pre: 2025-01 ~ 2025-02, post: 2025-03 ~ 2025-04
# 유의한 효과가 나오면 → 분석 설계에 문제 있음

if HAS_CI and len(weekly_ts) > 0:
    print("플라시보 테스트: 2025-03 가짜 개입")
    print("기대: 유의한 효과 없음")